In [1]:
%%writefile ../src/final_eval.py
"""Final test-set evaluation helpers. Used once, at the very end."""
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score


def cluster_bootstrap_ci(y, p, groups, metric, n_boot: int = 300, seed: int = 42):
    """95% interval for a metric, resampling whole customers (not single rows),
    because a customer's orders are not independent of each other."""
    y, p = np.asarray(y), np.asarray(p)
    idx = list(pd.Series(np.arange(len(y))).groupby(np.asarray(groups)).indices.values())
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_boot):
        pick = rng.integers(0, len(idx), len(idx))
        rows = np.concatenate([idx[i] for i in pick])
        vals.append(metric(y[rows], p[rows]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def _logloss(y, p):
    return log_loss(y, np.clip(p, 1e-6, 1 - 1e-6))


def scorecard_with_ci(y, p, groups, n_boot: int = 300) -> pd.DataFrame:
    rows = {}
    for name, fn in [("roc_auc", roc_auc_score), ("log_loss", _logloss),
                     ("brier", brier_score_loss)]:
        lo, hi = cluster_bootstrap_ci(y, p, groups, fn, n_boot)
        rows[name] = {"estimate": fn(np.asarray(y), np.asarray(p)),
                      "ci_low": lo, "ci_high": hi}
    return pd.DataFrame(rows).T.round(4)

Writing ../src/final_eval.py


In [2]:
import sys
sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import json, joblib, sklearn
import numpy as np, pandas as pd
from src import config
from src.features import split_feature_types
from src.models import make_logreg, make_hgb
from src.business import policy_result
from src.final_eval import scorecard_with_ci

train = pd.read_parquet(config.ROOT / "data/processed/train.parquet")
numeric, categorical = split_feature_types(train)
MODE = [config.SHIP_MODE]
all_feats = numeric + categorical

final = make_logreg(MODE, numeric).fit(train[MODE], train[config.TARGET])
challenger = make_hgb(all_feats, numeric, max_depth=3, learning_rate=0.05,
                      max_iter=200, early_stopping=False).fit(train[all_feats], train[config.TARGET])

config.MODELS_DIR.mkdir(exist_ok=True)
joblib.dump(final, config.MODELS_DIR / "final_mode_model.joblib")
joblib.dump(challenger, config.MODELS_DIR / "challenger_hgb.joblib")
json.dump({"features": MODE, "trained_on_rows": len(train),
           "sklearn": sklearn.__version__, "target": config.TARGET},
          open(config.MODELS_DIR / "final_mode_model_meta.json", "w"), indent=2)
print("Saved. Train rows:", len(train))

Saved. Train rows: 138213


In [3]:
test = pd.read_parquet(config.ROOT / "data/processed/test.parquet")
y_t, g_t = test[config.TARGET], test["Customer Id"]
print("Test:", test.shape, " late rate:", round(y_t.mean(), 4))

p_final = final.predict_proba(test[MODE])[:, 1]
p_chall = challenger.predict_proba(test[all_feats])[:, 1]

print("\n== FINAL MODEL: shipping mode only ==")
card_final = scorecard_with_ci(y_t, p_final, g_t)
print(card_final)

print("\n== CHALLENGER: boosted, all features (reported, not selected) ==")
card_chall = scorecard_with_ci(y_t, p_chall, g_t, n_boot=150)
print(card_chall)

print("\n== Calibration by shipping mode (test) ==")
cal = (test.assign(p=p_final).groupby(config.SHIP_MODE)
       .agg(orders=("p", "size"), predicted=("p", "mean"),
            observed=(config.TARGET, "mean")).round(4))
print(cal)

print("\n== Fixed policies on test ==")
policies = {"flag First+Second (threshold 0.5)": 0.5,
            "flag First+Second+Same Day (threshold 0.4)": 0.4}
pol_rows = {}
for name, thr in policies.items():
    flagged = p_final > thr
    pol_rows[name] = {"share_flagged": flagged.mean(),
                      "recall": y_t[flagged].sum() / y_t.sum(),
                      "precision": y_t[flagged].mean()}
print(pd.DataFrame(pol_rows).T.round(4))

Test: (34552, 38)  late rate: 0.5729

== FINAL MODEL: shipping mode only ==
          estimate  ci_low  ci_high
roc_auc     0.7445  0.7370   0.7518
log_loss    0.5332  0.5260   0.5402
brier       0.1863  0.1834   0.1893

== CHALLENGER: boosted, all features (reported, not selected) ==
          estimate  ci_low  ci_high
roc_auc     0.7378  0.7283   0.7470
log_loss    0.5372  0.5303   0.5439
brier       0.1880  0.1853   0.1910

== Calibration by shipping mode (test) ==
                orders  predicted  observed
Shipping Mode                              
First Class       5296     0.9997    1.0000
Same Day          1805     0.4816    0.4709
Second Class      6662     0.7947    0.8131
Standard Class   20789     0.3982    0.3960

== Fixed policies on test ==
                                            share_flagged  recall  precision
flag First+Second (threshold 0.5)                  0.3461  0.5412     0.8959
flag First+Second+Same Day (threshold 0.4)         0.3983  0.5841     0.8402


In [4]:
(config.ROOT / "reports").mkdir(exist_ok=True)
out = {"final": card_final.to_dict(), "challenger": card_chall.to_dict(),
       "calibration": cal.to_dict(), "policies": pd.DataFrame(pol_rows).T.round(4).to_dict()}
json.dump(out, open(config.ROOT / "reports/test_results.json", "w"), indent=2)
print("Saved reports/test_results.json")

Saved reports/test_results.json
